In [ ]:
import os
import re
import pandas as pd
import numpy as np
from Bio import Entrez
import time

# Always provide an email to NCBI so they can contact you if there are issues
Entrez.email = "melnik.varvara@gmail.com"

pd.set_option("display.max_columns", 200)

In [2]:
raw_data_path = "../data/raw/"

df_db = pd.read_csv(os.path.join(raw_data_path, 'protac.csv'), dtype=object)

print(df_db.shape)

(9380, 89)


In [3]:
# 1. Define only the columns needed for classification and identification
cols_for_analysis = [
    # Identifiers
    "Compound ID", "Smiles", "InChI Key", "Target", "E3 ligase",
    
    # Tier 1: Degradation
    "DC50 (nM)", "Dmax (%)", "Percent degradation (%)",
    
    # Tier 2: Ternary Complex
    "Kd (nM, Ternary complex)", "t1/2 (s, Ternary complex)", "delta G (kcal/mol, Ternary complex)",
    
    # Tier 3: Dual Binding (Binary)
    "EC50 (nM, Protac to Target)", "Kd (nM, Protac to Target)", "Ki (nM, Protac to Target)",
    "EC50 (nM, Protac to E3)", "Kd (nM, Protac to E3)", "Ki (nM, Protac to E3)"
]

# Filter to only existing columns in your df_db
df_db = df_db[[c for c in cols_for_analysis if c in df_db.columns]].copy()

In [4]:
def clean_protac_data(value):
    if pd.isna(value) or value == 'nan':
        return np.nan
    
    val_str = str(value).strip().lower()
    
    if val_str in ['n.d.', 'nd', 'n.a.', 'na', 'inactive']:
        return np.nan
    
    def safe_float(s):
        clean_p = re.sub(r'[^0-9.]', '', s)
        if not clean_p or clean_p == '.' or clean_p.count('.') > 1:
            return None
        try:
            return float(clean_p)
        except ValueError:
            return None

    # 1. Handle slashes (e.g., '91.4/100')
    if '/' in val_str:
        parts = val_str.split('/')
        nums = [safe_float(p) for p in parts if safe_float(p) is not None]
        return np.mean(nums) if nums else np.nan

    # 2. Handle ranges/approx (e.g., '10-30' or '1~3')
    if '-' in val_str or '~' in val_str:
        parts = re.split(r'-|~', val_str)
        nums = [safe_float(p) for p in parts if safe_float(p) is not None]
        return np.mean(nums) if nums else np.nan

    # 3. Handle single values and qualifiers (e.g., '>99', '..', '98.2')
    final_val = safe_float(val_str)
    return final_val if final_val is not None else np.nan

In [5]:
# List of columns containing numerical activity metrics
metrics_to_clean = [
    "DC50 (nM)", "Dmax (%)", "Percent degradation (%)",
    "Kd (nM, Ternary complex)", "t1/2 (s, Ternary complex)", "delta G (kcal/mol, Ternary complex)",
    "EC50 (nM, Protac to Target)", "Kd (nM, Protac to Target)", "Ki (nM, Protac to Target)",
    "EC50 (nM, Protac to E3)", "Kd (nM, Protac to E3)", "Ki (nM, Protac to E3)"
]

# Ensure we only try to clean columns that actually exist in the dataframe
metrics_present = [c for c in metrics_to_clean if c in df_db.columns]

for col in metrics_present:
    df_db[col] = df_db[col].apply(clean_protac_data)

In [6]:
df_db.head()

,Compound ID,Smiles,InChI Key,Target,E3 ligase,DC50 (nM),Dmax (%),Percent degradation (%),"Kd (nM, Ternary complex)","t1/2 (s, Ternary complex)","delta G (kcal/mol, Ternary complex)","EC50 (nM, Protac to Target)","Kd (nM, Protac to Target)","Ki (nM, Protac to Target)","EC50 (nM, Protac to E3)","Kd (nM, Protac to E3)","Ki (nM, Protac to E3)"
0,1,COC1=CC(C2=CN(C)C(=O)C3=CN=CC=C23)=CC(OC)=C1CN...,RPMQBLMPGMFXLD-PDUNVWSESA-N,BRD7,VHL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,COC1=CC(C2=CN(C)C(=O)C3=CN=CC=C23)=CC(OC)=C1CN...,RPMQBLMPGMFXLD-PDUNVWSESA-N,BRD9,VHL,NaN,NaN,NaN,85.5,NaN,9.7,NaN,15.0,NaN,NaN,28.5,NaN
2,76,CCN(CCOCCOCC(=O)N[C@H](C(=O)N1C[C@H](O)C[C@H]1...,DINLAPHIPWQULR-VQNWZXTLSA-N,ER,VHL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,173,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...,YPKQEFMVVKVSQU-RXJDTIOWSA-N,CRBN,VHL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,275,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...,ZSCOIFSUFMYZEZ-YSWDPXALSA-N,HER2,VHL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
def classify_protac(row):
    """
    Tiered classification: returns 1 if evidence of activity exists, 
    otherwise returns 0 (even for data-poor/missing rows).
    """
    # Tier 1: Functional Degradation (Potency & Efficacy)
    dc50 = row.get('DC50 (nM)')
    dmax = row.get('Dmax (%)')
    p_deg = row.get('Percent degradation (%)')
    
    if pd.notnull(dc50) and dc50 <= 1000:
        return 1
    if pd.notnull(dmax) and dmax >= 70:
        return 1
    if pd.notnull(p_deg) and p_deg >= 70:
        return 1

    # Tier 2: Ternary Complex Stability (Thermodynamics & Kinetics)
    kd_ternary = row.get('Kd (nM, Ternary complex)')
    t_half = row.get('t1/2 (s, Ternary complex)')
    delta_g = row.get('delta G (kcal/mol, Ternary complex)')

    if pd.notnull(kd_ternary) and kd_ternary <= 500:
        return 1
    if pd.notnull(t_half) and t_half >= 30:
        return 1
    if pd.notnull(delta_g) and delta_g <= -8:
        return 1

    # Tier 3: Dual Binding Proxy (Target AND E3 Affinities)
    # We look for any binding metric available (EC50, Kd, or Ki)
    t_metrics = ['EC50 (nM, Protac to Target)', 'Kd (nM, Protac to Target)', 'Ki (nM, Protac to Target)']
    e3_metrics = ['EC50 (nM, Protac to E3)', 'Kd (nM, Protac to E3)', 'Ki (nM, Protac to E3)']
    
    t_aff = next((row[c] for c in t_metrics if c in row and pd.notnull(row[c])), None)
    e3_aff = next((row[c] for c in e3_metrics if c in row and pd.notnull(row[c])), None)
    
    if t_aff is not None and e3_aff is not None:
        if t_aff <= 1000 and e3_aff <= 1000:
            return 1

    # DEFAULT: Return 0 if no evidence is met or row is empty
    return 0

In [8]:
# We want the most potent and effective values from across all sources
agg_rules = {
    'Compound ID': 'first',
    'Smiles': 'first',
    'Target': 'first',
    'E3 ligase': 'first',
    'DC50 (nM)': 'min',             
    'Dmax (%)': 'max',              
    'Percent degradation (%)': 'max',
    'Kd (nM, Ternary complex)': 'min',
    't1/2 (s, Ternary complex)': 'max', 
    'delta G (kcal/mol, Ternary complex)': 'min', 
    'EC50 (nM, Protac to Target)': 'min',
    'Kd (nM, Protac to Target)': 'min',
    'Ki (nM, Protac to Target)': 'min',
    'EC50 (nM, Protac to E3)': 'min',
    'Kd (nM, Protac to E3)': 'min',
    'Ki (nM, Protac to E3)': 'min'
}

# Filter rules to only include columns that actually exist in your df_db
final_agg = {k: v for k, v in agg_rules.items() if k in df_db.columns}

# Deduplicate by InChI Key, keeping the best-case values according to our defined aggregation rules
df_unique = df_db.groupby('InChI Key').agg(final_agg).reset_index()

# Assign activity labels to the unique molecules
df_unique['Activity_Label'] = df_unique.apply(classify_protac, axis=1)

print(f"Total entries in raw data: {len(df_db)}")
print(f"Unique molecules (InChI Keys): {len(df_unique)}")
print(f"Actives (1): {int(df_unique['Activity_Label'].sum())}")
print(f"Inactives (0): {len(df_unique) - int(df_unique['Activity_Label'].sum())}")
df_unique.head()

Total entries in raw data: 9380
Unique molecules (InChI Keys): 6109
Actives (1): 1407
Inactives (0): 4702


,InChI Key,Compound ID,Smiles,Target,E3 ligase,DC50 (nM),Dmax (%),Percent degradation (%),"Kd (nM, Ternary complex)","t1/2 (s, Ternary complex)","delta G (kcal/mol, Ternary complex)","EC50 (nM, Protac to Target)","Kd (nM, Protac to Target)","Ki (nM, Protac to Target)","EC50 (nM, Protac to E3)","Kd (nM, Protac to E3)","Ki (nM, Protac to E3)",Activity_Label
0,AAANIVVODIZQAG-NYJWCRJYSA-N,2868,COC1=CC(N2CCN(CCCCCCCCC(=O)N[C@H](C(=O)N3C[C@H...,EGFR del19/T790M/C797S,VHL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,AAEQMRGNIHSULB-UHFFFAOYSA-N,5077,NC1=NC=CC=C1C1=NC2=CC=C(C3=CC=CC(CCNC(=O)CNC4=...,AKT1,CRBN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,AAERNZFFOMNEGX-UHFFFAOYSA-N,4074,O=C(COC1=CC=CC2=C1C(=O)N(C1CCC(=O)NC1=O)C2=O)N...,CDK9,CRBN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,AAFBROZXFJRJFC-QNIQEMAOSA-N,3241,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...,WDR5,VHL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,AAGMQVFKPKFXDV-JMSRYGQUSA-N,4494,CC1=C(C2=CC=C([C@H](C)NC(=O)[C@@H]3C[C@@H](O)C...,PIP4K2C,VHL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,67.0,NaN,NaN,NaN,NaN,0


In [9]:
def is_entrez_accessible(target_name):
    """
    Checks if a target name returns results in the NCBI Gene database.
    """
    if pd.isna(target_name) or target_name == "":
        return False
    
    try:
        # Search for the target name in the Gene database
        handle = Entrez.esearch(db="gene", term=target_name)
        record = Entrez.read(handle)
        handle.close()
        
        # 'Count' tells us how many records matched the search
        count = int(record["Count"])
        
        # Respect NCBI rate limits (3 requests per second without API key)
        time.sleep(0.4) 
        
        return count > 0
    except Exception as e:
        print(f"Error checking {target_name}: {e}")
        return False

In [10]:
unique_targets = df_unique['Target'].dropna().unique()

target_validity = {target: is_entrez_accessible(target) for target in unique_targets}
df_unique['Entrez_Accessible'] = df_unique['Target'].map(target_validity)

accessible_count = df_unique['Entrez_Accessible'].sum()
print(f"Validation complete. {accessible_count} out of {len(df_unique)} rows have Entrez-accessible targets.")

Validation complete. 6087 out of 6109 rows have Entrez-accessible targets.


In [11]:
failed_targets_df = df_unique[df_unique['Entrez_Accessible'] == False]
failed_target_names = failed_targets_df['Target'].unique()

print(f"Unique failed target names ({len(failed_target_names)}):")
print(failed_target_names)

Unique failed target names (0):
[]


In [12]:
nan_targets_count = df_unique['Target'].isna().sum()
print(f"Rows with NaN Target: {nan_targets_count}")

print("\nBreakdown of Entrez_Accessible column:")
print(df_unique['Entrez_Accessible'].value_counts(dropna=False))

problem_rows = df_unique[
    (df_unique['Entrez_Accessible'] == False) | 
    (df_unique['Entrez_Accessible'].isna())
]

print(f"\nUnique values in 'Target' for problem rows:")
print(problem_rows['Target'].unique())

Rows with NaN Target: 22

Breakdown of Entrez_Accessible column:
Entrez_Accessible
True    6087
NaN       22
Name: count, dtype: int64

Unique values in 'Target' for problem rows:
[None]


In [13]:
df_final = df_unique[df_unique['Target'].notna()].copy()

output_path = os.path.join(raw_data_path, 'protac_cleaned.csv')
df_final.to_csv(output_path, index=False)